<a href="https://colab.research.google.com/github/ashaw23871/cnn-cifar10-pytorch/blob/main/CNN_for_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
import torchvision.transforms as transforms
transform = transforms.Compose([
    # center the data around 0 and ensure stable gradients
    transforms.ToTensor(),#[0,255] -> [0,1]
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [0,1] -> [-1,1] z = x-mean/std = 0-0.5/0.5 = -1
])

In [4]:
traniset = CIFAR10(root="/content/drive/MyDrive/Colab Notebooks/data", train=True, download = True, transform = transform)
testset = CIFAR10(root="/content/drive/MyDrive/Colab Notebooks/data",train=False, download = True, transform = transform)

In [5]:
from torch.utils.data import DataLoader
trainloader = DataLoader(traniset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build the CNN

In [6]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), # in_channel 3 for rgb, out_channel = no of unique filtres/ features maps
        nn.ReLU(),
        nn.MaxPool2d(2,2), # kernel = 2 , stride =2

        nn.Conv2d(32,64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1) # flatten
    x = self.fc_layers(x)
    return x

In [7]:
model = CNN()

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


# Training CNN

In [9]:
epochs = 10

for epoch in range(epochs):
  epoch_trainig_loss = 0.0

  for images, labels in trainloader:
    optimizer.zero_grad()
    outputs = model.forward(images)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()

    epoch_trainig_loss += loss.item()

  print(f"epoch {epoch+1}/{epochs} -> loss = {epoch_trainig_loss/len(trainloader)}")

epoch 1/10 -> loss = 1.384464647535168
epoch 2/10 -> loss = 0.9590583133042011
epoch 3/10 -> loss = 0.7797091062111623
epoch 4/10 -> loss = 0.6464850315276314
epoch 5/10 -> loss = 0.5470974202579855
epoch 6/10 -> loss = 0.4570059673979764
epoch 7/10 -> loss = 0.3780814129525743
epoch 8/10 -> loss = 0.2970380445994684
epoch 9/10 -> loss = 0.23991357067795208
epoch 10/10 -> loss = 0.1892690558148467


In [10]:
# Evaluate our CNN
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
  for images, labels in testloader:
    outputs = model.forward(images)
    _, predicted = torch.max(outputs,1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels *100} %")


accuracy = 75.91 %
